In [1]:
import os
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient
import pandas as pd

from dbrepo.api.dto import (
    QueryDefinition,
    FilterDefinition,
    FilterType,
    OrderDefinition,
    OrderType,
)

load_dotenv("../.env")

ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
USERNAME = os.getenv("DBREPO_USERNAME")
PASSWORD = os.getenv("DBREPO_PASSWORD")
DATABASE_ID = os.getenv("DBREPO_DATABASE_ID")

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("Current user:", client.whoami())
print("Database ID:", DATABASE_ID)

federicoa88
Current user: federicoa88
Database ID: bfa4385b-54a9-4ae3-b4f4-cb503d7bb016


In [2]:
# From T2_1
measurement_variables = pd.DataFrame([
    {"variable_code": "NS", "label": "Precipitation amount", "unit": "mm"},
    {"variable_code": "LF", "label": "Conductivity", "unit": "µS/cm"},
    {"variable_code": "pH", "label": "pH value", "unit": "dimensionless"},
    {"variable_code": "NH4", "label": "Ammonium concentration", "unit": "mg/L"},
    {"variable_code": "Na", "label": "Sodium concentration", "unit": "mg/L"},
    {"variable_code": "K", "label": "Potassium concentration", "unit": "mg/L"},
    {"variable_code": "Ca", "label": "Calcium concentration", "unit": "mg/L"},
    {"variable_code": "Mg", "label": "Magnesium concentration", "unit": "mg/L"},
    {"variable_code": "Cl", "label": "Chloride concentration", "unit": "mg/L"},
    {"variable_code": "NO3", "label": "Nitrate concentration", "unit": "mg/L"},
    {"variable_code": "SO4", "label": "Sulfate concentration", "unit": "mg/L"},
    {"variable_code": "Pb", "label": "Lead concentration", "unit": "µg/L"},
    {"variable_code": "Cd", "label": "Cadmium concentration", "unit": "µg/L"},
])

measurement_variables.insert(0, "variable_id", range(1, len(measurement_variables) + 1))

measurement_variables

,variable_id,variable_code,label,unit
0,1,NS,Precipitation amount,mm
1,2,LF,Conductivity,µS/cm
2,3,pH,pH value,dimensionless
3,4,NH4,Ammonium concentration,mg/L
4,5,Na,Sodium concentration,mg/L
5,6,K,Potassium concentration,mg/L
6,7,Ca,Calcium concentration,mg/L
7,8,Mg,Magnesium concentration,mg/L
8,9,Cl,Chloride concentration,mg/L
9,10,NO3,Nitrate concentration,mg/L


In [3]:
SOURCE_METADATA = {
    "original_title": "Concentrations of major ions in wet precipitation samples in Austria",
    "original_creators": "Peter Redl, Thomas Steinkogler, Anne Kasper-Giebl",
    "original_version": "1.0.0",
    "original_doi": "https://doi.org/10.48436/b0g4h-rv840",
    "original_repository": "TU Wien Research Data Repository",
    "original_license": "CC BY-NC-SA 4.0",
    "spatial_coverage": "Austria",
    "temporal_coverage": "2014-10-01 to 2020-12-30",
    "source_files": "precipitationdata.csv; stationcoordinates.csv"
}

## 8. Mapping units of measurement

This section describes the semantic annotation of the precipitation dataset by mapping all numeric variables to standardized units using controlled ontologies, and by aligning each variable with a formal unit definition. 

The primary reference ontology is the SI Digital Framework, which provides standardized representations of SI units. 

Each numeric variable physical quantity and associated unit are then mapped to corresponding ontology URIs, with preference given to SI-compliant representations. 


Where necessary, units are normalized into SI-coherent forms:
- mg/L and µg/L 
- electrical conductivity expressed in siemens per metre
- pH (dimensionless) explicitly marked

The resulting mappings are structured into a metadata payload that preserves both the original unit and its ontological representation. 

This metadata is then uploaded to DBRepo via its REST API


Example ![alt text](mm.png "Title")


### 8.1 Creating Metadata Ontology Entries for units

In [4]:
# Define a dictionary with the ontological mapping  SI Digital Framework
unit_ontology_map = {
    "mm": "http://si-digital-framework.org/SI/units/millimetre",
    "µS/cm": "http://si-digital-framework.org/SI/units/siemens-per-metre",
    "mg/L": "http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre",
    "µg/L": "http://si-digital-framework.org/SI/units/kilogram-per-cubic-metre",
    "dimensionless": "http://si-digital-framework.org/SI/units/unity"
}

# Mapping the unit to the corresponding ontology entry inside the measurement_variables dataframe
measurement_variables["ontology_unit"] = measurement_variables["unit"].map(unit_ontology_map)

In [5]:
measurement_variables_description = """
Measurement variable metadata derived from Redl, 
Steinkogler and Kasper-Giebl 2021, 
DOI: https://doi.org/10.48436/b0g4h-rv840, 
license: CC BY-NC-SA 4.0.

Semantic enrichment table for precipitation dataset
measurement variables.

Each variable is associated with:
- original unit of measurement
- SI-based ontology URI
- standardized semantic representation

Ontology mappings primarily follow the
SI Digital Framework.
"""



### Recreating measurement tables

In [6]:
# Removing previous table, recreate, update metadata description
### Uncomment to RUN

'''
client.delete_table(
    database_id=DATABASE_ID,
    name="measurement_variables",
)


print("Measurement variables table: DELETED. ")


# Recreate with newly created ontology mapping for units
measurement_variables_result = client.create_table(
    database_id=DATABASE_ID,
    name="measurement_variables",
    is_public=True,
    is_schema_public=True,
    dataframe=measurement_variables_for_dbrepo,
    description=measurement_variables_description
)

print("Measurement variables table re-created.")
print(measurement_variables_result)
'''

'\nclient.delete_table(\n    database_id=DATABASE_ID,\n    name="measurement_variables",\n)\n\n\nprint("Measurement variables table: DELETED. ")\n\n\n# Recreate with newly created ontology mapping for units\nmeasurement_variables_result = client.create_table(\n    database_id=DATABASE_ID,\n    name="measurement_variables",\n    is_public=True,\n    is_schema_public=True,\n    dataframe=measurement_variables_for_dbrepo,\n    description=measurement_variables_description\n)\n\nprint("Measurement variables table re-created.")\nprint(measurement_variables_result)\n'

In [7]:
# Rapid check for consistency
TABLE_STATIONS      = "53688cc1-5205-4f25-af30-7feef2ea1b2b"
TABLE_PRECIPITATION = "d11966e6-f0a6-460d-900b-b56e627fc752"
TABLE_VARIABLES     = "7ed509f5-3356-4318-80b3-c6672b13c4b8"

def get_column_ids(table_id):
    table = client.get_table(database_id=DATABASE_ID, table_id=table_id)
    return {col.name: col.id for col in table.columns}

stations_col_ids      = get_column_ids(TABLE_STATIONS)
precipitation_col_ids = get_column_ids(TABLE_PRECIPITATION)
variables_col_ids     = get_column_ids(TABLE_VARIABLES)



    

In [8]:
print(stations_col_ids)

{'station_id': '9f762d20-696c-477f-8b74-7f7ba052f579', 'station_code': '3ed0f861-e3ec-45e4-a48a-16fb8bf3852e', 'latitude': '2e160049-5b95-4435-b490-8e2e628e4c30', 'longitude': '0e0d51f7-55d3-4358-b355-dac60f65f8e1'}


## Checking Views 
Crate an SQL view where you select one single station, and extract Cd value 


```CREATE VIEW cd_station_id_7 AS
SELECT
    station_id,
    Cd
FROM precipitation_measurements
WHERE station_id = 'STATION_ID';```


In [9]:
#cabce3e7-11a1-45d1-9007-111a76b81d06 | fa_experiment_nb 
tables = client.get_tables(DATABASE_ID)
print(f"Tables in database: {len(tables)}")

for t in tables:
    print(f"- {t.id} | {t.name}")

Tables in database: 3
- 53688cc1-5205-4f25-af30-7feef2ea1b2b | stations
- 7ed509f5-3356-4318-80b3-c6672b13c4b8 | measurement_variables
- d11966e6-f0a6-460d-900b-b56e627fc752 | precipitation_measurements


In [26]:
TABLE_STATIONS_id       = "53688cc1-5205-4f25-af30-7feef2ea1b2b"
TABLE_PRECIPITATION_id  = "d11966e6-f0a6-460d-900b-b56e627fc752"
TABLE_VARIABLES_id      = "7ed509f5-3356-4318-80b3-c6672b13c4b8"

# Select Precipitation data for Cd with valid measurement flag and selected station_id


DATABASE_ID = os.getenv("DBREPO_DATABASE_ID")

table_precipitations = client.get_table(database_id=DATABASE_ID,
                                        table_id=TABLE_PRECIPITATION_id)

prec_df = client.get_table_data(database_id=DATABASE_ID,
                                table_id=TABLE_PRECIPITATION_id,
                                page=0,
                                size=1000000,)

stations_df = client.get_table_data(database_id=DATABASE_ID,
                                table_id=TABLE_STATIONS,
                                page=0,
                                size=1000000,)


TABLE_NAME = table_precipitations.name
print('Check TABLE_NAME :: ' , TABLE_NAME )
print('Check TABLE_PRECIPITATION_id :: ' , table_precipitations.id )


# Check columns from existing table
'''
precipitation_col_ids = get_column_ids(TABLE_PRECIPITATION_id)
print("\nprecipitation columns:")
for name, cid in precipitation_col_ids.items():
    print(f"  {name:20s} {cid}")


stations_col_ids = get_column_ids(TABLE_STATIONS_id)
print("\nprecipitation columns:")
for name, cid in stations_col_ids.items():
    print(f"  {name} {cid}")
'''

Check TABLE_NAME ::  precipitation_measurements
Check TABLE_PRECIPITATION_id ::  d11966e6-f0a6-460d-900b-b56e627fc752


'\nprecipitation_col_ids = get_column_ids(TABLE_PRECIPITATION_id)\nprint("\nprecipitation columns:")\nfor name, cid in precipitation_col_ids.items():\n    print(f"  {name:20s} {cid}")\n\n\nstations_col_ids = get_column_ids(TABLE_STATIONS_id)\nprint("\nprecipitation columns:")\nfor name, cid in stations_col_ids.items():\n    print(f"  {name} {cid}")\n'

In [27]:
# Getting precipitations data
prec_df = client.get_table_data(database_id=DATABASE_ID,
                                table_id=TABLE_PRECIPITATION_id,
                                page=0,
                                size=1000000,)

prec_df.head(2)

,ca_flag,cd_flag,cl,cl_flag,k_flag,lf,lf_flag,measurement_id,mg_flag,na_flag,...,sample_date,so4,so4_flag,station_id,ca,k,mg,na,cd,pb
0,7.0,7.0,0.05,1.0,7.0,8.3,1.0,3849,7.0,7.0,...,2017-05-06,0.24,1.0,3,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,7.0,0.131,1.0,1.0,12.33,1.0,2666,1.0,1.0,...,2016-06-20,0.609,1.0,7,0.233,0.024,0.021,0.19,NaN,NaN


In [ ]:
'''
  station_id	station_code	latitude	longitude
0	1	HF	47.470833	10.680833
1	2	NB	47.662222	12.226944
2	3	IV	46.818056	12.351667
3	4	NH	47.956389	13.016667
4	5	WW	47.421667	13.253333
'''
stations_df.head(2)

In [22]:
# Select Stations
TABLE_NAME = 'precipitation_measurements'
VALUE = '6' # HF

view_query = QueryDefinition(
    datasources=[TABLE_NAME],
    columns=[
        #f"{TABLE_NAME}.station_id",
        f"{TABLE_NAME}.cd",
        f"{TABLE_NAME}.cd_flag",
        f"{TABLE_NAME}.station_id",
        f"{TABLE_NAME}.pb_flag",
        f"{TABLE_NAME}.pb",
    ],

    filters=[
        FilterDefinition(
            type=FilterType.WHERE,
            column=f"{TABLE_NAME}.ph",
            operator=">=",
            value="7",
        ),
    ],

    orders=[
        OrderDefinition(
            column=f"{TABLE_NAME}.cd_flag",
            direction=OrderType.DESC,
        )
    ],
)

# Storing the view
view = client.create_view(
    database_id=DATABASE_ID,
    name=STATION_ID + "valid_measurements",
    query=view_query,
    is_public=False,
    is_schema_public=False,
)


QUERY.columns ['precipitation_measurements.cd', 'precipitation_measurements.cd_flag', 'precipitation_measurements.station_id', 'precipitation_measurements.pb_flag', 'precipitation_measurements.pb']
SELECT_COLUMNS :::  [SubsetColumn(id='97f27df3-c1da-4655-9c2d-9649d82c4e73', alias=None), SubsetColumn(id='782733ec-a92e-4917-b2b5-f8c3917c5f7f', alias=None), SubsetColumn(id='8e3bd238-7c59-44a4-b3f2-273f419faff3', alias=None), SubsetColumn(id='51641a87-61a9-4a85-b681-305930d38efb', alias=None), SubsetColumn(id='5941eb9e-92ea-4369-b446-2dfa74bbee09', alias=None)]
select_columns_fefi  [Column(id='97f27df3-c1da-4655-9c2d-9649d82c4e73', name='station_id', database_id='bfa4385b-54a9-4ae3-b4f4-cb503d7bb016', table_id='d11966e6-f0a6-460d-900b-b56e627fc752', ord=1, internal_name='station_id', is_null_allowed=False, type=<ColumnType.BIGINT: 'bigint'>, alias=None, description=None, size=None, d=None, mean=7.1392, median=7.1392, concept=None, unit=None, enums=[], sets=[], index_length=None, length=Non

ForbiddenError: Failed to create view: not allowed

In [18]:
VALUE = '2'
TABLE_NAME = 'stations'

In [19]:
view_query = QueryDefinition(
    
    datasources=[TABLE_NAME],
    
    columns=[
        f"{TABLE_NAME}.station_id",
        f"{TABLE_NAME}.latitude",
    ],
    
    filters=[
        
        FilterDefinition(
            type=FilterType.WHERE,
            column=f"{TABLE_NAME}.latitude",
            operator="=",
            value=VALUE,
        )
    ],
)


view = client.create_view(
    database_id=DATABASE_ID,
    name="stations_with_valid_cadmium_measurements",
    query=view_query,
    is_public=False,
    is_schema_public=False,
)


QUERY.columns ['stations.station_id', 'stations.latitude']
SELECT_COLUMNS :::  [SubsetColumn(id='9f762d20-696c-477f-8b74-7f7ba052f579', alias=None), SubsetColumn(id='2e160049-5b95-4435-b490-8e2e628e4c30', alias=None)]
select_columns_fefi  [Column(id='9f762d20-696c-477f-8b74-7f7ba052f579', name='station_id', database_id='bfa4385b-54a9-4ae3-b4f4-cb503d7bb016', table_id='53688cc1-5205-4f25-af30-7feef2ea1b2b', ord=0, internal_name='station_id', is_null_allowed=False, type=<ColumnType.BIGINT: 'bigint'>, alias=None, description=None, size=None, d=None, mean=7.5, median=7.5, concept=None, unit=None, enums=[], sets=[], index_length=None, length=None, data_length=None, max_data_length=None, num_rows=None, val_min=None, val_max=None, std_dev=4.0311), Column(id='2e160049-5b95-4435-b490-8e2e628e4c30', name='latitude', database_id='bfa4385b-54a9-4ae3-b4f4-cb503d7bb016', table_id='53688cc1-5205-4f25-af30-7feef2ea1b2b', ord=2, internal_name='latitude', is_null_allowed=False, type=<ColumnType.DECIMAL:

ForbiddenError: Failed to create view: not allowed

In [ ]:
print("Created view id:", view.id)
print("Created view name:", view.name)

VIEW_ID = view.id

### Accessing existing VIEWS

In [31]:
views = client.get_views(database_id = DATABASE_ID)

In [32]:
views

[]